In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

With some diffusiuon coefficent D
D = [10^-3 to 10^-2]

Variannce <L^2> =6D t^\alpha 3D
or <L^2> = 2*N*D*t^\alpha for any N dimension
Where alpha = 1 is pure diffusion (assume this one)
alpha = 2 is ballistic motion

We draw the step length from a gaussian of mean 0, with <L^2> variance at each time step
samnple random angle

Restrain it to a cell of a given size (around 5 microns in each dimension) perhaps with reflecting boundary


Run 1 experiment wher eparticle is moving, do max projection and log what we see
Keep particle running, and start capture in the same manner later. 

MOstly interested in seeing how the max projection is affected by this

In [12]:
# Define the 3D imaging volume
N = 20
xy_size = 0.16
z_size = 0.4
x_lim = (0, 512 * xy_size)
y_lim = (0, 512 * xy_size)
z_lim = (0, N * z_size)

nx = int(x_lim[1] / xy_size)
ny = int(y_lim[1] / xy_size)
nz = int(z_lim[1] / z_size)
x = np.arange(nx) * xy_size
y = np.arange(ny) * xy_size
z = np.arange(nz) * z_size
X, Y = np.meshgrid(x, y, indexing='ij')

# Particle starts at the center of the volume and travels in one random direction
r_0 = np.array([x_lim[1] / 2, y_lim[1] / 2, z_lim[1] / 2])
r_0 = np.array([x_lim[1] / 2, y_lim[1] / 2, 2])
dt = 1
velocity = 1
r_hat = np.random.normal(size=3)

r_hat = np.array([0, 1, 0])
r_hat = r_hat / np.linalg.norm(r_hat)
particle_radius = 0.5

# Acquire one XY plane at each z position, with dt seconds between acquisitions
scan_times = np.arange(nz) * dt
scan_positions = r_0 + scan_times[:, None] * velocity * r_hat
observed_slices = []
for zi, particle_position in zip(z, scan_positions):
    distance_squared = (
        (X - particle_position[0]) ** 2
        + (Y - particle_position[1]) ** 2
        + (zi - particle_position[2]) ** 2
    )
    observed_slices.append(distance_squared <= particle_radius ** 2)
observed_volume = np.stack(observed_slices)

from ipywidgets import interact, IntSlider

slices = observed_volume
vmax = 1

def show_slice(idx):
    plt.figure(figsize=(6, 6))
    plt.imshow(
        slices[idx].T,
        origin='lower',
        extent=(*x_lim, *y_lim),
        cmap='gray',
        vmin=0,
        vmax=vmax,
    )
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Observed z slice {idx} at t={scan_times[idx]:.1f} s')
    plt.colorbar()
    plt.show()

#interact(show_slice, idx=IntSlider(min=0, max=len(slices) - 1, step=1, value=0, description='Z slice'));

# Convert (z, x, y) array indices to physical (x, y, z) coordinates
observed_coords = np.argwhere(observed_volume)[:, [1, 2, 0]] * np.array([xy_size, xy_size, z_size])

# Interactive (rotatable) 3D view of the shape reconstructed from the moving z-stack
fig = go.Figure([
    go.Scatter3d(
        x=observed_coords[:, 0], y=observed_coords[:, 1], z=observed_coords[:, 2],
        mode='markers', marker=dict(size=4), name='Observed voxels',
    ),
    go.Scatter3d(
        x=scan_positions[:, 0], y=scan_positions[:, 1], z=scan_positions[:, 2],
        mode='lines', line=dict(color='red', width=4), name='Particle path during scan',
    ),
    go.Scatter3d(
        x=[r_0[0]], y=[r_0[1]], z=[r_0[2]],
        mode='markers', marker=dict(size=8, color='green'), name='Starting position',
    ),
])
fig.update_layout(
    title='Observed shape from sequential z-stack',
    width=800, height=800,
    scene=dict(
        xaxis=dict(title='x', range=list(x_lim)),
        yaxis=dict(title='y', range=list(y_lim)),
        zaxis=dict(title='z', range=list(z_lim)),
        aspectmode='manual',
        aspectratio=dict(x=x_lim[1], y=y_lim[1], z=z_lim[1]),
        camera=dict(eye=dict(x=80, y=80, z=7)),
    ),
)
fig.show()